### Can we make a tensor value class , initially using numpy  ? 
If in future you ever face numerical error in the outputs, then it may be due to uneven floating point precision 

In [ ]:
import matplotlib.pyplot as plt 
import math 
import random 

In [ ]:
import math 
import numpy as np

class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.asarray(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._prev = set(_children)
        self._op = _op 
        self._backward = lambda : None
        
    def __repr__(self): 
        return f'Tensor(shape={self.data.shape}, dtype={self.data.dtype})'

    
    @staticmethod
    def unbroadcast(grad, original_shape): 
        grad_shape = grad.shape  
        ones_to_add = len(grad.shape) - len(original_shape)
        temp_shape = ones_to_add * (1,) + original_shape

        temp_shape = np.asarray(temp_shape)
        grad_shape = np.asarray(grad_shape)
        axes = np.where((temp_shape == 1) & (grad_shape != 1))[0]

        if len(axes) == 0: 
            return grad 
        return grad.sum(axis=tuple(axes), keepdims=True).reshape(original_shape)

    
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other) 
            
        t = self.data + other.data 
        out = Tensor(t , (self, other), _op='+')

        def _backward():
            self.grad += Tensor.unbroadcast(out.grad, self.data.shape)
            other.grad += Tensor.unbroadcast(out.grad , other.data.shape)

        out._backward = _backward
        return out 

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        
        t = self.data * other.data 
        out = Tensor(t, (self, other), _op='*') 

        def _backward(): 
            self.grad += Tensor.unbroadcast(
                out.grad * other.data,
                self.data.shape
            )
            
            other.grad += Tensor.unbroadcast(
                out.grad * self.data, 
                other.data.shape
            )

        out._backward = _backward
        return out 

    def sum(self, axis=None, keepdims=False): 
        if axis is None: 
            axes = None 
        elif isinstance(axis, tuple): 
            axes = axis 
        else: 
            axes = (axis,)
        
        t = np.sum(self.data, axis=axes, keepdims=keepdims)
        out = Tensor(t, (self,), 'sum')

        def _backward():
            grad = out.grad 
            if axis is not None and not keepdims:
                for ax in sorted(axes): 
                    grad = np.expand_dims(grad, axis=ax) # this just adds 1 to the missing dimensions 
            self.grad += np.broadcast_to(grad , self.data.shape) # this scales the 1 to correct dimension
            return  

        out._backward = _backward
        return out  

    def mean(self, axis=None, keepdims=False):
        if axis is None:
            num = self.data.size
        elif isinstance(axis, tuple):
            num = np.prod([self.data.shape[ax] for ax in axis])
        else:
            num = self.data.shape[axis]
    
        return self.sum(axis=axis, keepdims=keepdims) / num

    def exp(self): 
        t = np.exp(self.data) 

        out = Tensor(t , (self, ), 'exp')

        def _backward():
            self.grad += t * out.grad
            return  

        out._backward = _backward
        return out 

    def relu(self): 
        t = np.maximum(0, self.data)
        
        out = Tensor(t , (self,) , 'ReLU')

        def _backward():
            self.grad += (self.data > 0) * out.grad 
            return  

        out._backward = _backward
        return out 

    def tanh(self): 
        t = np.tanh(self.data)
        out = Tensor(t , (self, ), 'tanh' )

        def _backward(): 
            self.grad += (1 - t**2) * out.grad 

        out._backward = _backward
        return out 

    def log(self): 
        t = np.log(self.data + 1e-12) 

        out = Tensor(t , (self,), 'log')

        def _backward(): 
            self.grad += (1.0/(self.data + 1e-12)) * out.grad 
            return 

        out._backward = _backward 
        return out 

    def sigmoid(self): 
        a = np.exp(-self.data)
        t = 1 / (1.0 + a)

        out = Tensor(t, (self,), 'sigmoid')
        
        def _backward():
            self.grad += out.data * (1.0 - out.data) * out.grad
            return 
            
        out._backward = _backward
        return out 

    # def __matmul__(self, other): 
    #     other = other if isinstance(other, Tensor) else Tensor(other)
        
    #     assert self.data.ndim == 2
    #     assert other.data.ndim == 2
        
    #     t = np.matmul(self.data, other.data)
    #     out = Tensor(t , (self, other), 'matmul')

    #     def _backward():
    #         self.grad += np.matmul(out.grad ,np.transpose(other.data))
    #         other.grad += np.matmul(np.transpose(self.data) ,out.grad)
             
    #     out._backward = _backward
    #     return out 

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
    
        A = self.data
        B = other.data
    
        assert A.ndim >= 2
        assert B.ndim >= 2
    
        # Matrix dimensions must align
        assert A.shape[-1] == B.shape[-2]
    
        t = np.matmul(A, B)
    
        out = Tensor(t, (self, other), "matmul")
    
        def _backward():
    
            dA = np.matmul(
                out.grad,
                np.swapaxes(B, -1, -2)
            )
    
            dB = np.matmul(
                np.swapaxes(A, -1, -2),
                out.grad
            )
    
            self.grad += Tensor.unbroadcast(
                dA,
                self.data.shape
            )
    
            other.grad += Tensor.unbroadcast(
                dB,
                other.data.shape
            )
    
        out._backward = _backward
    
        return out        

    def __neg__(self): 
        return self * -1

    def __sub__(self, other): 
        other = other if isinstance(other, Tensor) else Tensor(other)

        return self + -other 

    def __pow__(self, other): 
        assert isinstance(other , (int, float))

        t = (self.data) ** other 
        out = Tensor(t , (self,), f'pow{other}')

        def _backward():
            self.grad += Tensor.unbroadcast(other * (self.data ** (other - 1)) * out.grad , self.data.shape) 
            
        out._backward = _backward
        return out 

    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self * (other ** -1)

    def __radd__(self, other): 
        return self + other 

    def __rsub__(self, other): 
        return -self + other 

    def __rtruediv__(self, other): 
        return other * (self ** -1)

    def __rmul__(self, other):
        return self * other

    def __len__(self):
        return len(self.data)
        
    def reshape(self, new_shape): 
        out = Tensor(np.reshape(self.data, new_shape), (self,), 'reshape')

        def _backward(): 
            self.grad += out.grad.reshape(self.data.shape)
            return
        out._backward = _backward
        return out 

    def transpose(self):
        out = Tensor(self.data.T, (self,), 'transpose')

        def _backward():
            self.grad += out.grad.T 
            return 
        out._backward = _backward
        return out 

    def sqrt(self): 
        t = np.sqrt(self.data + 1e-12)
        out = Tensor(t, (self,), 'sqrt')

        def _backward():
            self.grad += (1/(2 * t)) * out.grad
        out._backward = _backward
        return out 

    def abs(self): 
        t = np.abs(self.data) 
        out = Tensor(t, (self,), 'abs')

        def _backward(): 
            self.grad += np.sign(self.data) * out.grad 
            return 
        out._backward = _backward
        return out 

    def clip(self, a , b): 
        t = np.clip(self.data , a , b)
        out = Tensor(t, (self,), 'clip')

        def _backward(): 
            mask = (self.data >= a) & (self.data <= b)
            self.grad += mask * out.grad 
            
        out._backward = _backward
        return out 
    
    def __getitem__(self, index): 
        out = Tensor(self.data[index], (self,), "getitem")

        def _backward():
            grad = np.zeros_like(self.data)
            np.add.at(
                grad, 
                index, 
                out.grad 
            ) 
            self.grad += grad 

        out._backward = _backward
        return out 

    def max(self, axis=None, keepdims=False):
        if axis is None:
            axes = None
        elif isinstance(axis, tuple):
            axes = axis
        else:
            axes = (axis,)
    
        t = np.max(self.data, axis=axes, keepdims=keepdims)
        out = Tensor(t, (self,), "max")
    
        def _backward():
            grad = out.grad
            max_values = t
    
            if axis is not None and not keepdims:
                for ax in sorted(axes):
                    grad = np.expand_dims(grad, axis=ax)
                    max_values = np.expand_dims(max_values, axis=ax)
    
            mask = (self.data == max_values)
    
            count = mask.sum(axis=axis, keepdims=keepdims)
    
            if axis is not None and not keepdims:
                for ax in sorted(axes):
                    count = np.expand_dims(count, axis=ax)
    
            grad = np.broadcast_to(grad, self.data.shape)
            count = np.broadcast_to(count, self.data.shape)
    
            self.grad += mask * grad / count
    
        out._backward = _backward
        return out

    def softmax(self, axis=-1, keepdims=True):
        shifted = self - self.max(axis=axis, keepdims=True)
        exp_x = shifted.exp() 
        return exp_x / exp_x.sum(axis=axis, keepdims=True)
        
    def squeeze(self, axis=None):
        out = Tensor(
            np.squeeze(self.data, axis=axis), 
            (self,), 
            "squeeze"
        )

        def _backward():
            self.grad += out.grad.reshape(self.data.shape)
        out._backward = _backward 

        return out 

    def unsqueeze(self, axis): 
        out = Tensor(
            np.expand_dims(self.data, axis=axis),
            (self,), 
            "unsqueeze"
        )

        def _backward(): 
            self.grad += np.squeeze(out.grad, axis=axis) 
        out._backward = _backward

        return out 

    def logsumexp(self, axis=-1, keepdims=False): 
        max_x = self.max(axis=axis, keepdims=True)

        shifted = self - max_x 

        out = shifted.exp().sum(
            axis=axis, 
            keepdims=True 
        ).log() + max_x 

        if not keepdims: 
            out = out.squeeze(axis) 

        return out 

    def log_softmax(self, axis=-1): 
        
        return self - self.logsumexp(
            axis=axis, 
            keepdims=True 
        ) 
    
    @property
    def T(self):
        return self.transpose()

    @property
    def shape(self): 
        return self.data.shape 

    @property
    def ndim(self):
        return self.data.ndim 
        
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

In [ ]:
class Parameter(Tensor):
    '''Marks the tensor as learnable'''
    pass 

In [ ]:
class Module:

    def parameters(self):
        params = []

        for value in self.__dict__.values():
            if isinstance(value, Parameter):
                params.append(value)

            elif isinstance(value, Module):
                params.extend(value.parameters())

            elif isinstance(value, (list, tuple)): 
                for item in value: 
                    if isinstance(item, Parameter):
                        params.append(item)

                    elif isinstance(item, Module): 
                        params.extend(item.parameters())
                
        return params

In [ ]:
class Linear(Module):
    '''
    X : (batch, in_features)
    W : (in_features, out_features)
    b : (out_features,)
    Y : (batch, out_features)

    Y = XW + b 
    '''
    def __init__(self, in_features, out_features): 

        std = np.sqrt(2.0/in_features)
        
        self.W = Parameter(
            np.random.randn(in_features, out_features) * std 
        )

        self.b = Parameter(
            np.zeros(out_features)
        )

    def __call__(self, x):
        return x @ self.W + self.b 

In [ ]:
class Sequential(Module):

    def __init__(self, *modules):
        self.modules = modules

    def __call__(self, x): 
        for module in self.modules: 
            x = module(x)
        return x 

In [ ]:
class ReLU(Module): 

    def __call__(self, x):
        return x.relu()

In [ ]:
class Sigmoid(Module): 

    def __call__(self, x): 
        return x.sigmoid()

In [ ]:
class Tanh(Module): 

    def __call__(self, x): 
        return x.tanh()

In [ ]:
class MSELoss(Module): 

    def __call__(self, predictions, target): 
        return ((predictions - target)**2).mean()

In [ ]:
class MAELoss(Module): 

    def __call__(self, predictions, target): 
        return (predictions - target).abs().mean() 

In [ ]:
class BCELoss(Module): 

    def __call__(self, predictions, target): 
        eps = 1e-12 

        predictions = predictions.clip(eps, 1.0 - eps) 

        return - (
            target * (predictions.log())
            + 
            (1 - target) * (1 - predictions).log()
        ).mean()

In [ ]:
class CrossEntropyLoss(Module): 

    def __call__(self, logits, target): 
        log_probs = logits.log_softmax(axis=-1)

        batch_indices = np.arange(len(target)) 

        correct_log_probs = log_probs[
            batch_indices,
            target
        ]

        return -correct_log_probs.mean()

Optimizer
│
├── SGD
├── Momentum
├── Adam
└── AdamW

In [ ]:
class Optimizer: 

    def __init__(self, parameters): 
        self.parameters = list(parameters)

    def zero_grad(self): 
        for p in self.parameters: 
            p.grad.fill(0)

    def step(self): 
        raise NotImplementedError

In [ ]:
class SGD(Optimizer):  

    def __init__(self, parameters, lr=0.01): 
        super().__init__(parameters)
        self.lr = lr 

    def step(self): 
        for p in self.parameters: 
            p.data -= self.lr * p.grad 

In [ ]:
class Momentum(Optimizer):

    def __init__(
        self, 
        parameters, 
        lr=0.01, 
        momentum=0.9
    ): 

        super().__init__(parameters)

        self.lr = lr 
        self.momentum = momentum 

        self.velocity = [
            np.zeros_like(p.data)
            for p in self.parameters
        ]

    def step(self): 
        for p, v in zip(self.parameters, self.velocity): 

            v *= self.momentum 
            v += p.grad 

            p.data -= self.lr * v 

In [ ]:
class Adam(Optimizer): 

    def __init__(
        self, 
        parameters, 
        lr = 0.001, 
        beta1 = 0.9, 
        beta2 = 0.999, 
        eps = 1e-8 
    ): 
        super().__init__(parameters)
        self.lr = lr 
        self.beta1 = beta1 
        self.beta2 = beta2 
        self.eps = eps 
        self.t = 0  

        self.m = [
            np.zeros_like(p.data) 
            for p in self.parameters
        ]

        self.v = [
            np.zeros_like(p.data) 
            for p in self.parameters
        ]

       
    def step(self): 

        self.t += 1 
        
        for p, m, v in zip(self.parameters, self.m, self.v):

            # First Moment 
            m *= self.beta1
            m += (1.0 - self.beta1) * p.grad 

            # Second Moment 
            v *= self.beta2
            v += (1.0 - self.beta2) * (p.grad ** 2)

            # Bias correction 
            m_cap = m/(1.0 - self.beta1 ** self.t)
            v_cap = v/(1.0 - self.beta2 ** self.t) 

            # Parameter update 
            p.data -= self.lr * m_cap/ (np.sqrt(v_cap) + self.eps)

In [ ]:
class AdamW(Optimizer):

    def __init__(
        self, 
        parameters, 
        lr = 0.001, 
        beta1 = 0.9, 
        beta2 = 0.999,
        weight_decay = 0.01, 
        eps = 1e-8 
    ): 
        super().__init__(parameters)
        self.lr = lr 
        self.beta1 = beta1 
        self.beta2 = beta2 
        self.weight_decay = weight_decay
        self.eps = eps 
        self.t = 0  

        self.m = [
            np.zeros_like(p.data) 
            for p in self.parameters
        ]

        self.v = [
            np.zeros_like(p.data) 
            for p in self.parameters
        ]


    def step(self): 
        self.t += 1 
        
        for p, m, v in zip(self.parameters, self.m, self.v):

            # First Moment 
            m *= self.beta1
            m += (1.0 - self.beta1) * p.grad 

            # Second Moment 
            v *= self.beta2
            v += (1.0 - self.beta2) * (p.grad ** 2)

            # Bias correction 
            m_cap = m/(1.0 - self.beta1 ** self.t)
            v_cap = v/(1.0 - self.beta2 ** self.t) 

            update = m_cap/ (np.sqrt(v_cap) + self.eps)

            # Decoupled weight decay 
            update += self.weight_decay * p.data 

            # Parameter update 
            p.data -= self.lr * update

In [ ]:
class DataLoader:

    def __init__(self, X, y, batch_size, shuffle=False): 
        self.X = X 
        self.y = y 
        self.batch_size = batch_size
        self.shuffle = shuffle 


    def __iter__(self): 
        n = len(self.X)

        indices = np.arange(n)

        if self.shuffle: 
            np.random.shuffle(indices)


        for start in range(0, n, self.batch_size): 
            batch_indices = indices[start: start+self.batch_size]

            yield(
                self.X[batch_indices], 
                self.y[batch_indices]
            )

In [ ]:
def numerical_grad(f , x : np.ndarray , eps = 1e-5): 
    
    numericGrad = np.zeros_like(x)

    it = np.nditer(
        x, 
        flags=['multi_index'],
        op_flags=['readwrite']
    )

    while not it.finished: 

        idx = it.multi_index
        original = x[idx]

        x[idx] = original + eps 
        y_plus = f(x)

        x[idx] = original - eps 
        y_minus = f(x)

        numericGrad[idx] = (y_plus - y_minus)/(2 * eps)
        x[idx] = original

        it.iternext()

    return numericGrad

In [ ]:
def check_gradient(f, x_data, eps=1e-5):

    # ---------------------
    # Analytical gradient
    # ---------------------

    x = Tensor(x_data.copy())

    y = f(x)

    assert y.data.size == 1, \
        "f(x) must return a scalar"

    y.backward()

    analytical = x.grad.copy()


    # ---------------------
    # Numerical gradient
    # ---------------------

    numerical = numerical_grad(
        lambda data: f(Tensor(data)).data,
        x_data.copy(),
        eps
    )


    # ---------------------
    # Compare
    # ---------------------

    difference = np.abs(
        analytical - numerical
    )

    print("Analytical:")
    print(analytical)

    print("\nNumerical:")
    print(numerical)

    print("\nMaximum difference:")
    print(difference.max())

    return np.allclose(
        analytical,
        numerical,
        rtol=1e-4,
        atol=1e-6
    )